In [39]:
import utils
import importlib

importlib.reload(utils)

<module 'utils' from '/home/jovyan/_shared_storage/temp_stud/MA_Otto/utils.py'>

### Test Feature Matrix Creation Function
- All tests successful, if output matrix has dtype float64. Precision errors for float32

In [5]:
import unittest
import numpy as np
import pandas as pd

from data_utils import compute_feature_target_matrix, PRICE_MEASURES

import unittest
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

SEC = 1_000_000_000          # 1 second in nanoseconds
MS100 = 100_000_000          # 100 ms in nanoseconds


def make_df(timestamps_ns, price=None, imbalance=None):
    """Build a minimal input DataFrame with controllable values."""
    n = len(timestamps_ns)
    return pd.DataFrame({
        "ts":         timestamps_ns,
        "MicroPrice_tick-based_10_1s": price      if price      is not None else np.arange(1.0, n + 1),
        "L1-QImb":   imbalance  if imbalance  is not None else np.zeros(n),
    })


# ---------------------------------------------------------------------------
# Test suite
# ---------------------------------------------------------------------------

class TestResamplingLogic(unittest.TestCase):
    """Verify that 100 ms binning keeps only the last tick per bucket
    and that empty buckets are dropped."""

    def _run(self, timestamps_ns, expected_bins):
        df = make_df(timestamps_ns)
        result = compute_feature_target_matrix(
            df, ts_col="ts",
            target_cols=[], feature_cols=[],
            horizons=["1s"],
            resample_flag=True,
        )
        np.testing.assert_array_equal(result["bin"].to_numpy(), expected_bins)

    def test_two_ticks_same_bucket_keeps_last(self):
        """Two ticks in the same 100 ms window → one output row (the later tick)."""
        ts = np.array([0, 50_000_000], dtype=np.int64)   # both in [0, 100ms)
        result = compute_feature_target_matrix(
            make_df(ts, price=np.array([10.0, 20.0])),
            ts_col="ts", target_cols=["MicroPrice_tick-based_10_1s"], feature_cols=[],
            horizons=["1s"], resample_flag=True,
        )
        self.assertEqual(len(result), 1)
        self.assertAlmostEqual(result["MicroPrice_tick-based_10_1s"].iloc[0], 20.0)

    def test_ticks_in_separate_buckets_both_kept(self):
        """Ticks 200 ms apart land in different buckets → two output rows."""
        ts = np.array([0, 200_000_000], dtype=np.int64)
        result = compute_feature_target_matrix(
            make_df(ts), ts_col="ts",
            target_cols=[], feature_cols=[],
            horizons=["1s"], resample_flag=True,
        )
        self.assertEqual(len(result), 2)

    def test_empty_bucket_omitted(self):
        """A 300 ms gap creates no phantom row for the missing bucket."""
        ts = np.array([0, 300_000_000], dtype=np.int64)
        result = compute_feature_target_matrix(
            make_df(ts), ts_col="ts",
            target_cols=[], feature_cols=[],
            horizons=["1s"], resample_flag=True,
        )
        self.assertEqual(len(result), 2)
        bins = result["bin"].to_numpy()
        self.assertNotIn(MS100, bins)   # the 100 ms bucket should not exist

    def test_bin_column_values_are_100ms_multiples(self):
        """Every bin value must be an exact multiple of 100 ms."""
        ts = np.array([0, 50_000_000, 150_000_000, 250_000_000], dtype=np.int64)
        result = compute_feature_target_matrix(
            make_df(ts), ts_col="ts",
            target_cols=[], feature_cols=[],
            horizons=["1s"], resample_flag=True,
        )
        self.assertTrue(np.all(result["bin"].to_numpy() % MS100 == 0))


class TestTimestampConsistency(unittest.TestCase):
    """Output rows must be aligned with the resampled/input timestamps."""

    def test_output_row_count_matches_unique_bins(self):
        ts = np.array([0, 0, MS100, MS100, 2 * MS100], dtype=np.int64)
        result = compute_feature_target_matrix(
            make_df(ts), ts_col="ts",
            target_cols=[], feature_cols=[],
            horizons=["1s"], resample_flag=True,
        )
        self.assertEqual(len(result), 3)

    def test_no_resample_preserves_all_rows(self):
        ts = np.arange(5, dtype=np.int64) * SEC
        result = compute_feature_target_matrix(
            make_df(ts), ts_col="ts",
            target_cols=[], feature_cols=[],
            horizons=["1s"], resample_flag=False,
        )
        self.assertEqual(len(result), 5)

    def test_output_index_is_contiguous(self):
        """After resampling the DataFrame index must be a clean RangeIndex."""
        ts = np.arange(4, dtype=np.int64) * MS100
        result = compute_feature_target_matrix(
            make_df(ts), ts_col="ts",
            target_cols=[], feature_cols=[],
            horizons=["1s"], resample_flag=True,
        )
        expected = pd.RangeIndex(len(result))
        pd.testing.assert_index_equal(result.index, expected)


class TestForwardLogReturns(unittest.TestCase):
    """T_{target}_LogReturn_{h} = log( price[t+h] / price[t] )."""

    def setUp(self):
        # Evenly spaced ticks at 1 s intervals, no resampling needed.
        n = 6
        self.ts = np.arange(n, dtype=np.int64) * SEC
        self.prices = np.array([100.0, 110.0, 121.0, 133.1, 146.41, 161.05])
        self.df = make_df(self.ts, price=self.prices)

    def _result(self, horizons):
        return compute_feature_target_matrix(
            self.df.copy(), ts_col="ts",
            target_cols=["MicroPrice_tick-based_10_1s"], feature_cols=[],
            horizons=horizons, resample_flag=False,
        )

    def test_1s_forward_return_middle_rows(self):
        """Each row's 1 s return equals log(price[i+1] / price[i])."""
        result = self._result(["1s"])
        col = "T_MicroPrice_tick-based_10_1s_LogReturn_1s"
        for i in range(len(self.prices) - 1):
            expected = np.log(self.prices[i + 1] / self.prices[i])
            self.assertAlmostEqual(result[col].iloc[i], expected, places=10)

    def test_2s_forward_return(self):
        """2 s horizon skips one row: log(price[i+2] / price[i])."""
        result = self._result(["2s"])
        col = "T_MicroPrice_tick-based_10_1s_LogReturn_2s"
        for i in range(len(self.prices) - 2):
            expected = np.log(self.prices[i + 2] / self.prices[i])
            self.assertAlmostEqual(result[col].iloc[i], expected, places=10)

    def test_last_rows_are_nan_when_horizon_exceeds_data(self):
        """Rows near the end of the series have no future tick → NaN."""
        result = self._result(["1s"])
        col = "T_MicroPrice_tick-based_10_1s_LogReturn_1s"
        self.assertTrue(np.isnan(result[col].iloc[-1]))

    def test_two_second_horizon_last_two_rows_are_nan(self):
        result = self._result(["2s"])
        col = "T_MicroPrice_tick-based_10_1s_LogReturn_2s"
        self.assertTrue(np.isnan(result[col].iloc[-1]))
        self.assertTrue(np.isnan(result[col].iloc[-2]))

    def test_zero_horizon_return_is_zero(self):
        """A 0 s horizon compares a price with itself → log(1) = 0."""
        result = self._result(["0s"])
        col = "T_MicroPrice_tick-based_10_1s_LogReturn_0s"
        np.testing.assert_array_almost_equal(
            result[col].dropna().to_numpy(), 0.0
        )


class TestBackwardFeatureLags(unittest.TestCase):
    """
    Feature windows are non-overlapping.
    For horizons ["-1s", "-2s"] (sorted to [-2s, -1s]):
      k=0 (-2s): window = [t-2s, t-1s]
      k=1 (-1s): window = [t-1s, t]      (last horizon → end = current row)
    """

    def setUp(self):
        n = 6
        self.ts = np.arange(n, dtype=np.int64) * SEC
        self.prices   = np.array([1.0, 2.0, 4.0, 8.0, 16.0, 32.0])
        self.imbalance = np.array([0.0, 1.0, 3.0, 6.0, 10.0, 15.0])
        self.df = make_df(self.ts, price=self.prices, imbalance=self.imbalance)

    def _result(self, horizons):
        return compute_feature_target_matrix(
            self.df.copy(), ts_col="ts",
            target_cols=[], feature_cols=["MicroPrice_tick-based_10_1s", "L1-QImb"],
            horizons=horizons, resample_flag=False,
        )

    def test_price_feature_nearest_lag(self):
        """F_MicroPrice_tick-based_10_1s_-1s at row i = log(price[i] / price[i-1])."""
        result = self._result(["-1s", "-2s"])
        col = "F_MicroPrice_tick-based_10_1s_-1s"
        for i in range(1, len(self.prices)):
            expected = np.log(self.prices[i] / self.prices[i - 1])
            self.assertAlmostEqual(result[col].iloc[i], expected, places=10)

    def test_price_feature_farther_lag(self):
        """F_MicroPrice_tick-based_10_1s_-2s at row i = log(price[i-1] / price[i-2]) (non-overlapping)."""
        result = self._result(["-1s", "-2s"])
        col = "F_MicroPrice_tick-based_10_1s_-2s"
        # k=0: window [t-2s, t-1s] → log(price[i-1] / price[i-2])
        for i in range(2, len(self.prices)):
            expected = np.log(self.prices[i - 1] / self.prices[i - 2])
            self.assertAlmostEqual(result[col].iloc[i], expected, places=10)

    def test_price_feature_windows_are_non_overlapping(self):
        """-1s and -2s windows together cover [t-2s, t] without overlap."""
        result = self._result(["-1s", "-2s"])
        i = 3  # safe interior row
        r_near = result["F_MicroPrice_tick-based_10_1s_-1s"].iloc[i]   # [t-1s, t]
        r_far  = result["F_MicroPrice_tick-based_10_1s_-2s"].iloc[i]   # [t-2s, t-1s]
        combined = np.log(self.prices[i] / self.prices[i - 2])
        self.assertAlmostEqual(r_near + r_far, combined, places=10)

    # -- Difference features (L1-QImb is not in PRICE_MEASURES) -------------

    def test_imbalance_feature_nearest_lag(self):
        """F_L1-QImb_-1s at row i = imbalance[i] - imbalance[i-1]."""
        result = self._result(["-1s", "-2s"])
        col = "F_L1-QImb_-1s"
        for i in range(1, len(self.imbalance)):
            expected = self.imbalance[i] - self.imbalance[i - 1]
            self.assertAlmostEqual(result[col].iloc[i], expected, places=10)

    def test_imbalance_feature_farther_lag(self):
        """F_L1-QImb_-2s at row i = imbalance[i-1] - imbalance[i-2]."""
        result = self._result(["-1s", "-2s"])
        col = "F_L1-QImb_-2s"
        for i in range(2, len(self.imbalance)):
            expected = self.imbalance[i - 1] - self.imbalance[i - 2]
            self.assertAlmostEqual(result[col].iloc[i], expected, places=10)

    # -- NaN boundary behavior -----------------------------------------------

    def test_early_rows_are_nan_when_lag_exceeds_history(self):
        """Row 0 has no history at all → every backward feature is NaN."""
        result = self._result(["-1s"])
        col = "F_MicroPrice_tick-based_10_1s_-1s"
        self.assertTrue(np.isnan(result[col].iloc[0]))

    def test_single_lag_horizon_nan_boundary(self):
        """With only -1s, row 0 is NaN, row 1 onward is valid."""
        result = self._result(["-1s"])
        col = "F_MicroPrice_tick-based_10_1s_-1s"
        self.assertFalse(np.isnan(result[col].iloc[1]))


class TestMixedHorizons(unittest.TestCase):
    """Sanity check with both forward and backward horizons present."""

    def test_forward_and_backward_columns_both_present(self):
        n = 5
        ts = np.arange(n, dtype=np.int64) * SEC
        df = make_df(ts)
        result = compute_feature_target_matrix(
            df, ts_col="ts",
            target_cols=["MicroPrice_tick-based_10_1s"],
            feature_cols=["MicroPrice_tick-based_10_1s", "L1-QImb"],
            horizons=["1s", "-1s"],
            resample_flag=False,
        )
        self.assertIn("T_MicroPrice_tick-based_10_1s_LogReturn_1s",  result.columns)
        self.assertIn("F_MicroPrice_tick-based_10_1s_-1s",           result.columns)
        self.assertIn("F_L1-QImb_-1s",              result.columns)

    def test_input_columns_preserved_in_output(self):
        """All original columns must still be present in the result."""
        n = 4
        ts = np.arange(n, dtype=np.int64) * SEC
        df = make_df(ts)
        result = compute_feature_target_matrix(
            df, ts_col="ts",
            target_cols=["MicroPrice_tick-based_10_1s"], feature_cols=["L1-QImb"],
            horizons=["1s", "-1s"], resample_flag=False,
        )
        for col in df.columns:
            self.assertIn(col, result.columns)

if __name__ == "__main__":
    unittest.main(
        argv=['first-arg-is-ignored'],
        exit=False,
        verbosity=2)

test_early_rows_are_nan_when_lag_exceeds_history (__main__.TestBackwardFeatureLags.test_early_rows_are_nan_when_lag_exceeds_history)
Row 0 has no history at all → every backward feature is NaN. ... ok
test_imbalance_feature_farther_lag (__main__.TestBackwardFeatureLags.test_imbalance_feature_farther_lag)
F_L1-QImb_-2s at row i = imbalance[i-1] - imbalance[i-2]. ... ok
test_imbalance_feature_nearest_lag (__main__.TestBackwardFeatureLags.test_imbalance_feature_nearest_lag)
F_L1-QImb_-1s at row i = imbalance[i] - imbalance[i-1]. ... ok
test_price_feature_farther_lag (__main__.TestBackwardFeatureLags.test_price_feature_farther_lag)
F_MicroPrice_tick-based_10_1s_-2s at row i = log(price[i-1] / price[i-2]) (non-overlapping). ... ok
test_price_feature_nearest_lag (__main__.TestBackwardFeatureLags.test_price_feature_nearest_lag)
F_MicroPrice_tick-based_10_1s_-1s at row i = log(price[i] / price[i-1]). ... ok
test_price_feature_windows_are_non_overlapping (__main__.TestBackwardFeatureLags.test_p